# Antibiotic Prescribing Clustering Analysis
## Phase 2a: Quantitative Analysis Pipeline

| | |
|---|---|
| **Study** | Paediatric Outpatient Antibiotic Prescribing Archetypes |
| **Institution** | Lady Ridgeway Hospital for Children, Colombo |
| **Researcher** | Dr. Chinthaka Jayarathne, MD Health Informatics Batch 05, PGIM |
| **University** | University of Colombo |
| **Data** | January 2026, HHIMS pseudonymised dataset (17,345 encounters) |
| **Method** | K-medoids clustering with Gower distance |

---

## How to Use
1. **Install**: Run the *Install Dependencies* cell once (restart kernel if prompted)
2. **Configure**: Set `PATH_TO_DATA` in the *Configuration* cell to your CSV file
3. **Execute**: Run cells sequentially, or use *Kernel → Restart & Run All*

## Pipeline Steps

| Step | Description | Key Output |
|------|-------------|------------|
| 1 | Data Preparation & EDA | `encounter_level_clean.csv` + 5 figures |
| 2 | Prescriber Behaviour Analysis | `encounter_level_with_prescriber.csv` + 3 figures |
| 3 | K-medoids Clustering (Gower) | `encounter_level_clustered.csv` + 5 figures |
| 4 | Archetype Interpretation | `cluster_analysis_report.docx` |
| 5 | Bootstrap Validation | `validation_results.xlsx` + 2 figures |
| 6 | Final Deliverables | Excel workbook + PowerPoint + Word report |

> **Runtime**: Step 3 (Gower matrix on 17 k rows) ~5–15 min · Step 5 (bootstrap) ~10–20 min

## Cell 1 — Install Dependencies

> Run once. Restart the kernel if prompted.

In [ ]:
# Install required packages (run once; restart kernel if prompted)
import subprocess, sys

_packages = [
    "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "gower", "kmedoids",
    "python-docx", "openpyxl", "python-pptx", "scipy",
]
for _pkg in _packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg, "-q"])
print("All packages ready.")

## Cell 2 — Configuration

> **Edit `PATH_TO_DATA`** to point to your CSV file before running.

In [ ]:
import os

# ============================================================
# CONFIGURATION — set your data file path here
# ============================================================
PATH_TO_DATA = "D:/Academic/MD Research 2025/raw data/2026_januaryPrescriptions_pseudonymised.csv"

# Output directories (created automatically relative to this notebook)
BASE_DIR    = os.getcwd()
OUTPUT_DIR  = os.path.join(BASE_DIR, "output")
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
REPORTS_DIR = os.path.join(OUTPUT_DIR, "reports")
DATA_DIR    = os.path.join(OUTPUT_DIR, "data")

# Intermediate CSV paths
ENCOUNTER_CLEAN_PATH      = os.path.join(DATA_DIR, "encounter_level_clean.csv")
ENCOUNTER_PRESCRIBER_PATH = os.path.join(DATA_DIR, "encounter_level_with_prescriber.csv")
ENCOUNTER_CLUSTERED_PATH  = os.path.join(DATA_DIR, "encounter_level_clustered.csv")

# Age strata: (lower_months_inclusive, upper_months_exclusive, label)
AGE_STRATA = [
    (0,   1,    "neonate_0_27d"),
    (1,   3,    "infant_1_2m"),
    (3,   6,    "infant_3_5m"),
    (6,   12,   "infant_6_11m"),
    (12,  24,   "toddler_12_23m"),
    (24,  60,   "preschool_2_4y"),
    (60,  144,  "child_5_11y"),
    (144, 216,  "adolescent_12_17y"),
    (216, 9999, "adult_18plus"),
]

# Data-cleaning thresholds
MIN_AGE_MONTHS,  MAX_AGE_MONTHS = 0, 216
MIN_WEIGHT_KG,   MAX_WEIGHT_KG  = 0, 150

# Clustering parameters
K_RANGE                   = range(3, 9)   # k = 3 to 8
BOOTSTRAP_ITERATIONS      = 20
BOOTSTRAP_SAMPLE_FRACTION = 0.80
BOOTSTRAP_MAX_SAMPLE      = 3000          # cap rows per iteration

# Prescribing threshold
POLYPHARMACY_THRESHOLD = 3

# Create output folder structure
for _d in [OUTPUT_DIR, FIGURES_DIR, REPORTS_DIR, DATA_DIR]:
    os.makedirs(_d, exist_ok=True)

print("Configuration loaded.")
print(f"  Data   : {PATH_TO_DATA}")
print(f"  Output : {OUTPUT_DIR}")

## Cell 3 — Library Imports

In [ ]:
import re, os, time, warnings
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import gower
import kmedoids as km_lib
from scipy import stats

from docx import Document
from docx.shared import Inches as DocInches, Pt as DocPt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print("Libraries imported.")

---
## Step 1 — Data Preparation & Exploratory Analysis

- Loads the raw HHIMS CSV (encounter-level, pseudonymised)
- Renames pseudonymised ID columns
- Derives `antibiotic_monotherapy`, `antibiotic_combination`, `polypharmacy_flag`
- Cleans age / weight outliers
- Harmonises the free-text `Complaint` field into 15 binary symptom flags
- Prints EDA statistics
- Saves **5 PNG figures** to `output/figures/`

**Output file**: `output/data/encounter_level_clean.csv`

In [ ]:
"""
Step 1: Data Preparation & Exploratory Data Analysis
=====================================================
Data is already encounter-level from SQL extraction.
This script:
- Loads the CSV
- Adds remaining derived variables (monotherapy, combination, polypharmacy)
- Cleans age/weight outliers
- Prints EDA statistics
- Creates visualizations
- Saves encounter_level_clean.csv
"""
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")

def load_data(path):
    """Load the encounter-level CSV."""
    print("=" * 60)
    print("STEP 1: DATA PREPARATION & EXPLORATORY ANALYSIS")
    print("=" * 60)

    df = pd.read_csv(path, low_memory=False)

    # Normalize column names for pseudonymised data
    rename_map = {
        "patient_pseudo_id": "patient_id",
        "prescriber_pseudo_id": "prescriber_id",
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

    print(f"\nLoaded: {df.shape[0]} encounters, {df.shape[1]} columns")
    print(f"Columns: {list(df.columns)}")
    print(f"\nFirst 3 rows:")
    print(df.head(3).to_string())
    print(f"\nData types:\n{df.dtypes}")
    return df

def add_derived_variables(df):
    """Add antibiotic_monotherapy, antibiotic_combination, polypharmacy_flag."""
    print("\n--- Adding derived variables ---")

    df["antibiotic_monotherapy"] = (df["num_antibiotics"] == 1).astype(int)
    df["antibiotic_combination"] = (df["num_antibiotics"] >= 2).astype(int)
    df["polypharmacy_flag"] = (
        df["num_distinct_drugs"] >= POLYPHARMACY_THRESHOLD
    ).astype(int)

    print(f"  antibiotic_monotherapy: {df['antibiotic_monotherapy'].sum()} encounters")
    print(f"  antibiotic_combination: {df['antibiotic_combination'].sum()} encounters")
    print(f"  polypharmacy_flag: {df['polypharmacy_flag'].sum()} encounters")
    return df

def clean_data(df):
    """Apply data cleaning rules for age and weight."""
    print("\n--- Cleaning data ---")
    n_before = len(df)

    # Parse DateTimeOfVisit
    df["DateTimeOfVisit"] = pd.to_datetime(df["DateTimeOfVisit"], errors="coerce")

    # Age cleaning
    mask_age = (df["age_months"] >= MIN_AGE_MONTHS) & (df["age_months"] <= MAX_AGE_MONTHS)
    n_age_removed = (~mask_age).sum()
    df = df[mask_age].copy()
    print(f"  Removed {n_age_removed} encounters with invalid age")

    # Weight cleaning: set invalid to NaN, keep missing as NaN
    invalid_weight = (
        (df["PatientWeight"] < MIN_WEIGHT_KG) | (df["PatientWeight"] > MAX_WEIGHT_KG)
    )
    n_invalid_wt = invalid_weight.sum()
    df.loc[invalid_weight, "PatientWeight"] = np.nan
    print(f"  Set {n_invalid_wt} invalid weight values to NaN")
    print(f"  Weight missing/NaN: {df['PatientWeight'].isna().sum()} encounters")

    # Standardize drug_groups text
    if "drug_groups" in df.columns:
        df["drug_groups"] = df["drug_groups"].astype(str).str.strip()

    print(f"  Final dataset: {len(df)} encounters (removed {n_before - len(df)})")
    return df

def print_eda(df):
    """Print exploratory statistics."""
    print("\n" + "=" * 60)
    print("EXPLORATORY DATA ANALYSIS")
    print("=" * 60)

    total = len(df)
    ab_n = df["has_antibiotic"].sum()
    ab_rate = ab_n / total * 100

    print(f"\nTotal encounters:          {total}")
    print(f"Unique patients:           {df['patient_id'].nunique()}")
    print(f"Unique prescribers:        {df['prescriber_id'].nunique()}")
    print(f"Date range:                {df['DateTimeOfVisit'].min()} to {df['DateTimeOfVisit'].max()}")
    print(f"\nAntibiotic rate:           {ab_rate:.1f}% ({ab_n}/{total})")
    print(f"Monotherapy rate (of AB):  {df['antibiotic_monotherapy'].sum()}/{ab_n} = "
          f"{df['antibiotic_monotherapy'].sum() / ab_n * 100:.1f}%" if ab_n > 0 else "N/A")
    print(f"Combination rate (of AB):  {df['antibiotic_combination'].sum()}/{ab_n} = "
          f"{df['antibiotic_combination'].sum() / ab_n * 100:.1f}%" if ab_n > 0 else "N/A")
    print(f"Polypharmacy rate:         {df['polypharmacy_flag'].mean() * 100:.1f}%")
    print(f"Mean drugs/encounter:      {df['num_distinct_drugs'].mean():.2f}")

    print("\n--- Antibiotic rate by age stratum ---")
    age_order = [s[2] for s in AGE_STRATA]
    age_ab = df.groupby("age_stratum").agg(
        n=("OPDID", "count"),
        ab_rate=("has_antibiotic", "mean"),
    ).reindex(age_order).dropna()
    age_ab["ab_rate_pct"] = (age_ab["ab_rate"] * 100).round(1)
    print(age_ab[["n", "ab_rate_pct"]].to_string())

    print("\n--- Distribution of num_distinct_drugs ---")
    print(df["num_distinct_drugs"].describe().to_string())

def create_visualizations(df):
    """Create and save EDA charts."""
    print("\n--- Creating visualizations ---")

    age_order = [s[2] for s in AGE_STRATA]

    # 1. Encounters by age group
    fig, ax = plt.subplots(figsize=(10, 6))
    counts = df["age_stratum"].value_counts().reindex(age_order).dropna()
    counts.plot(kind="bar", ax=ax, color="steelblue", edgecolor="black")
    ax.set_title("Number of Encounters by Age Group", fontsize=14)
    ax.set_xlabel("Age Stratum")
    ax.set_ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/01_encounters_by_age.png", dpi=150)
    plt.close()

    # 2. Antibiotic pie chart
    fig, ax = plt.subplots(figsize=(8, 8))
    ab_counts = df["has_antibiotic"].value_counts().sort_index()
    ax.pie(ab_counts, labels=["No Antibiotic", "Has Antibiotic"],
           colors=["#66b3ff", "#ff6666"], autopct="%1.1f%%",
           startangle=90, textprops={"fontsize": 12})
    ax.set_title("Antibiotic vs Non-Antibiotic Encounters", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/02_antibiotic_pie.png", dpi=150)
    plt.close()

    # 3. Drugs per encounter histogram
    fig, ax = plt.subplots(figsize=(10, 6))
    max_drugs = int(df["num_distinct_drugs"].max())
    df["num_distinct_drugs"].plot(
        kind="hist", bins=range(1, max_drugs + 2),
        ax=ax, color="steelblue", edgecolor="black", align="left"
    )
    ax.set_title("Distribution of Drugs per Encounter", fontsize=14)
    ax.set_xlabel("Number of Distinct Drugs")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/03_drugs_histogram.png", dpi=150)
    plt.close()

    # 4. Antibiotics by age stratum (box plot)
    fig, ax = plt.subplots(figsize=(12, 6))
    ab_data = df[df["has_antibiotic"] == 1]
    if len(ab_data) > 0:
        present = [a for a in age_order if a in ab_data["age_stratum"].values]
        sns.boxplot(data=ab_data, x="age_stratum", y="num_antibiotics",
                    order=present, ax=ax)
    ax.set_title("Antibiotics per Encounter by Age (AB encounters only)", fontsize=14)
    ax.set_xlabel("Age Stratum")
    ax.set_ylabel("Number of Antibiotics")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/04_antibiotics_by_age_box.png", dpi=150)
    plt.close()

    # 5. Antibiotic rate by age stratum (bar)
    fig, ax = plt.subplots(figsize=(10, 6))
    ab_by_age = df.groupby("age_stratum")["has_antibiotic"].mean().reindex(age_order).dropna() * 100
    ab_by_age.plot(kind="bar", ax=ax, color="salmon", edgecolor="black")
    ax.set_title("Antibiotic Prescribing Rate by Age Group", fontsize=14)
    ax.set_xlabel("Age Stratum")
    ax.set_ylabel("Antibiotic Rate (%)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/05_ab_rate_by_age.png", dpi=150)
    plt.close()

    print(f"  Saved 5 figures to {FIGURES_DIR}/")

def _detect_symptoms(raw):
    """Return a frozenset of canonical symptom tags from a raw Complaint string.

    Key rules:
    - A single letter that is an entire whitespace/punctuation-delimited token
      maps to its symptom (c->COUGH, f->FEVER, v->VOMITING, r->RASH).
    - A letter embedded inside a multi-char token (e.g. 'fc', 'ccf') is part of
      a compound abbreviation, not a standalone symptom.
    - Duration markers (1d, 2d, d1, today, etc.) are stripped before matching.
    """
    if pd.isna(raw) or not str(raw).strip():
        return frozenset()

    t = str(raw).lower().strip()
    # Normalise punctuation to spaces
    t = re.sub(r'[,;/()\[\].]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()

    # Strip duration markers so they don't interfere with letter detection
    # e.g. "1d","2d","3d","1w","1wk","d1","d2","x1d","1/52","today"
    t = re.sub(r'\b(x\s*)?\d+\s*(d|days?|w|wks?|weeks?|m|months?)\b', ' ', t)
    t = re.sub(r'\bd\d+\b', ' ', t)       # d0, d1, d2 …
    t = re.sub(r'\b\d+/52\b', ' ', t)     # 1/52, 2/52 …
    t = re.sub(r'\btoday\b|\bthis morning\b|\bnow\b', ' ', t)
    t = re.sub(r'\b\d+\b', ' ', t)        # isolated numbers (c 2, c 3, f 3 …)
    t = re.sub(r'\s+', ' ', t).strip()

    s = set()

    # ── A. Compound multi-char abbreviations (before single-char checks) ──────
    # ccf / fcc = cough + cold + fever
    if re.search(r'\bccf\b|\bfcc\b', t):       s.update({'COUGH', 'COLD', 'FEVER'})
    # ccw = cough + cold + wheezing
    if re.search(r'\bccw\b', t):               s.update({'COUGH', 'COLD', 'WHEEZING'})
    # cc alone = cough + cold
    if re.search(r'\bcc\b', t):               s.update({'COUGH', 'COLD'})
    # cf = cough + fever
    if re.search(r'\bcf\b', t):               s.update({'COUGH', 'FEVER'})
    # fc = fever + cough
    if re.search(r'\bfc\b', t):               s.update({'FEVER', 'COUGH'})
    # vf / fv = vomiting + fever
    if re.search(r'\bvf\b|\bfv\b', t):        s.update({'VOMITING', 'FEVER'})
    # "c c" or "c , c" (two standalone c-tokens) = cough + cold
    if re.search(r'\bc\b.*\bc\b', t):         s.update({'COUGH', 'COLD'})
    # "f c" style (fever + cough as separate tokens)
    if re.search(r'\bf\b.*\bc\b', t):         s.update({'FEVER', 'COUGH'})

    # ── B. COUGH ─────────────────────────────────────────────────────────────
    if re.search(r'\bc\b', t):                s.add('COUGH')   # standalone c
    if re.search(r'\bcou', t):                s.add('COUGH')   # cough + all typos
    if re.search(r'\bcaugh\b|\bcopugh\b|\bcoifgh\b|\bcoiugh\b|\bcvough\b', t):
        s.add('COUGH')

    # ── C. COLD / URTI / RHINITIS ────────────────────────────────────────────
    if re.search(r'\bcold\b', t):             s.add('COLD')
    if re.search(r'\burti\b|\brti\b', t):     s.update({'COUGH', 'COLD'})
    if re.search(r'nasopharyngitis|common cold', t): s.add('COLD')
    if re.search(r'\brhinitis\b|\bnasal\b|\bsneezing\b', t): s.add('COLD')

    # ── D. FEVER ─────────────────────────────────────────────────────────────
    if re.search(r'\bf\b', t):                s.add('FEVER')   # standalone f
    if re.search(r'\bfever', t):              s.add('FEVER')   # fever, feverr, feveer
    if re.search(r'\bfev\b|\bfe\b|\bfe\d', t): s.add('FEVER')
    if re.search(r'\bf\d', t):               s.add('FEVER')   # f1, f1d, f2, f3

    # ── E. WHEEZING / SOB ────────────────────────────────────────────────────
    if re.search(r'\bwheez', t):              s.add('WHEEZING')
    if re.search(r'\bsob\b', t):              s.add('SHORTNESS_OF_BREATH')

    # ── F. VOMITING ──────────────────────────────────────────────────────────
    if re.search(r'\bv\b', t):                s.add('VOMITING')   # standalone v
    if re.search(r'\bvomit|\bvomi\b|\bvom\b', t): s.add('VOMITING')

    # ── G. DIARRHOEA ─────────────────────────────────────────────────────────
    if re.search(r'\bdiarr|\bloose\b|\bloos\b', t): s.add('DIARRHOEA')

    # ── H. ABDOMINAL PAIN ────────────────────────────────────────────────────
    if re.search(r'\babd\b|\babdominal\b|\bepigastric\b|\bab\s*pain\b|\bupper abd\b', t):
        s.add('ABDOMINAL_PAIN')

    # ── I. RASH / SKIN CONDITIONS ────────────────────────────────────────────
    if re.search(r'\brash|\brashes|\brasg\b|\brsh\b|\brasj\b', t): s.add('RASH')
    if re.search(r'\burticaria|\burticarial\b', t):                 s.add('URTICARIA')
    if re.search(r'\bscabi', t):              s.add('SCABIES')
    if re.search(r'\bimpetigo\b', t):         s.add('IMPETIGO')
    if re.search(r'\btinea\b|\btenia\b|\btenea\b|\bpityriasis|\bpytariasis|\bpytraisis', t):
        s.add('TINEA')
    if re.search(r'\beczema\b|\bdermatitis\b', t): s.add('DERMATITIS_ECZEMA')
    if re.search(r'\bfungal\b', t):           s.add('FUNGAL_SKIN')
    if re.search(r'\bchickenpox\b|\bchicken pox\b', t): s.add('CHICKENPOX')
    if re.search(r'\bhfmd\b|\bhand foot\b', t): s.add('HFMD')
    if re.search(r'\bpustul', t):             s.add('PUSTULAR')

    # ── J. TRAUMA / INJURY ───────────────────────────────────────────────────
    if re.search(r'\bfall\b|\btrauma\b|\brta\b|\binjury\b|\bwound\b|\bblunt\b'
                 r'|\bimpact\b|\blaceration\b|\bsprain\b|\bprick\b|\bcut\b', t):
        s.add('TRAUMA')
    if re.search(r'\bbite\b|\bscratch\b|\bdog\b|\brat\b|\bcat\b', t):
        s.add('ANIMAL_BITE')

    # ── K. WORMS ─────────────────────────────────────────────────────────────
    if re.search(r'\bworm', t):               s.add('WORMS')

    # ── L. EAR ───────────────────────────────────────────────────────────────
    if re.search(r'\bear\b|\bearache\b|\bear pain\b|\bear discharge\b', t):
        s.add('EAR_COMPLAINT')

    # ── M. EYE ───────────────────────────────────────────────────────────────
    if re.search(r'\beye\b|\bred eye\b', t):  s.add('EYE_COMPLAINT')

    # ── N. SORE THROAT ───────────────────────────────────────────────────────
    if re.search(r'\bsore\s*throat\b|\bthroat\s*pain\b|\bsore\s*tr\b|\bthr\s*p\b', t):
        s.add('SORE_THROAT')

    # ── O. OTHER COMMON COMPLAINTS ───────────────────────────────────────────
    if re.search(r'\bheadache\b|\bhead ache\b', t):  s.add('HEADACHE')
    if re.search(r'\bconstip', t):            s.add('CONSTIPATION')
    if re.search(r'\buti\b|\bdysuri|\burinary\b', t): s.add('UTI')
    if re.search(r'\ballerg', t):             s.add('ALLERGY')
    if re.search(r'\bchest pain\b', t):       s.add('CHEST_PAIN')
    if re.search(r'\bscalp\b|\bdandr', t):    s.add('SCALP_COMPLAINT')
    if re.search(r'\btooth|\btoothache\b', t): s.add('DENTAL')
    if re.search(r'\bloa\b|\bloss of appetite\b', t): s.add('LOSS_OF_APPETITE')
    if re.search(r'\blymph\b|\bcervical ln\b', t):    s.add('LYMPHADENOPATHY')
    if re.search(r'\bjoint pain\b|\bleg pain\b|\blimb pain\b|\bback\s*ache\b'
                 r'|\bneck pain\b|\bankle\b', t):     s.add('MUSCULOSKELETAL_PAIN')
    if re.search(r'\bphimosis\b|\bphymosis\b', t):    s.add('PHIMOSIS')
    if re.search(r'\babscess\b', t):          s.add('ABSCESS')

    return frozenset(s)

def harmonize_complaints(df):
    """Add complaint_harmonized and binary symp_* columns.

    complaint_harmonized: sorted semicolon-separated list of canonical tags.
    symp_*: binary (0/1) flags for the 15 clinically most relevant categories.
    """
    print("\n--- Harmonizing Complaint field ---")

    syms_series = df['Complaint'].map(_detect_symptoms)

    # Harmonized text tag string
    df['complaint_harmonized'] = syms_series.map(
        lambda s: '; '.join(sorted(s)) if s else 'UNSPECIFIED'
    )

    # Binary symptom columns
    BINARY_SYMS = {
        'symp_cough':      lambda s: 'COUGH' in s,
        'symp_cold':       lambda s: 'COLD' in s,
        'symp_fever':      lambda s: 'FEVER' in s,
        'symp_wheeze_sob': lambda s: bool(s & {'WHEEZING', 'SHORTNESS_OF_BREATH'}),
        'symp_vomiting':   lambda s: 'VOMITING' in s,
        'symp_diarrhoea':  lambda s: 'DIARRHOEA' in s,
        'symp_rash':       lambda s: bool(s & {'RASH', 'URTICARIA', 'IMPETIGO',
                                               'TINEA', 'DERMATITIS_ECZEMA',
                                               'FUNGAL_SKIN', 'CHICKENPOX',
                                               'HFMD', 'PUSTULAR'}),
        'symp_abdom_pain': lambda s: 'ABDOMINAL_PAIN' in s,
        'symp_trauma':     lambda s: bool(s & {'TRAUMA', 'ANIMAL_BITE'}),
        'symp_scabies':    lambda s: 'SCABIES' in s,
        'symp_worms':      lambda s: 'WORMS' in s,
        'symp_ear':        lambda s: 'EAR_COMPLAINT' in s,
        'symp_eye':        lambda s: 'EYE_COMPLAINT' in s,
        'symp_uti':        lambda s: 'UTI' in s,
        'symp_urti':       lambda s: bool(s & {'COUGH', 'COLD'}),
    }

    for col, fn in BINARY_SYMS.items():
        df[col] = syms_series.map(fn).astype(int)

    print(f"  Unique harmonized complaint combinations: "
          f"{df['complaint_harmonized'].nunique()}")
    print("  Symptom prevalence:")
    for col in BINARY_SYMS:
        n = df[col].sum()
        print(f"    {col}: {n} ({n / len(df) * 100:.1f}%)")

    return df

def run():
    """Execute Step 1."""
    df = load_data(PATH_TO_DATA)
    df = add_derived_variables(df)
    df = clean_data(df)
    df = harmonize_complaints(df)
    print_eda(df)
    create_visualizations(df)
    df.to_csv(ENCOUNTER_CLEAN_PATH, index=False)
    print(f"\nSaved: {ENCOUNTER_CLEAN_PATH}")
    return df

# ── Run Step 1 ──
run()

---
## Step 2 — Prescriber Behaviour Analysis

- Computes per-prescriber antibiotic rate, polypharmacy rate, mean drugs/encounter
- Merges prescriber-level metrics back onto each encounter row
- Saves **3 PNG figures** to `output/figures/`

**Output file**: `output/data/encounter_level_with_prescriber.csv`

In [ ]:
"""
Step 2: Prescriber Behaviour Analysis
=======================================
- Calculate prescriber-level metrics
- Merge back to encounter level
- Visualize prescriber behaviour
- Save encounter_level_with_prescriber.csv
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")

def run():
    print("=" * 60)
    print("STEP 2: PRESCRIBER BEHAVIOUR ANALYSIS")
    print("=" * 60)

    df = pd.read_csv(ENCOUNTER_CLEAN_PATH, low_memory=False)
    print(f"\nLoaded: {len(df)} encounters")
    print(f"Unique prescribers: {df['prescriber_id'].nunique()}")

    # ------------------------------------------------------------------
    # Calculate prescriber-level metrics
    # ------------------------------------------------------------------
    print("\n--- Computing prescriber metrics ---")
    prescriber = df.groupby("prescriber_id").agg(
        prescriber_total_encounters=("OPDID", "count"),
        prescriber_ab_encounters=("has_antibiotic", "sum"),
        prescriber_sum_antibiotics=("num_antibiotics", "sum"),
        prescriber_polypharmacy_encounters=("polypharmacy_flag", "sum"),
    ).reset_index()

    prescriber["prescriber_antibiotic_rate"] = (
        prescriber["prescriber_ab_encounters"] / prescriber["prescriber_total_encounters"]
    ).round(4)
    prescriber["prescriber_mean_num_antibiotics"] = (
        prescriber["prescriber_sum_antibiotics"] / prescriber["prescriber_total_encounters"]
    ).round(4)
    prescriber["prescriber_polypharmacy_rate"] = (
        prescriber["prescriber_polypharmacy_encounters"] / prescriber["prescriber_total_encounters"]
    ).round(4)

    print(f"\nPrescriber summary statistics:")
    print(prescriber[[
        "prescriber_antibiotic_rate",
        "prescriber_mean_num_antibiotics",
        "prescriber_polypharmacy_rate",
    ]].describe().to_string())

    # ------------------------------------------------------------------
    # Top 10 prescribers by volume
    # ------------------------------------------------------------------
    print("\n--- Top 10 prescribers by volume ---")
    top10 = prescriber.nlargest(10, "prescriber_total_encounters")[[
        "prescriber_id", "prescriber_total_encounters",
        "prescriber_antibiotic_rate", "prescriber_polypharmacy_rate"
    ]]
    print(top10.to_string(index=False))

    # ------------------------------------------------------------------
    # Merge prescriber metrics to encounter level
    # ------------------------------------------------------------------
    merge_cols = [
        "prescriber_id",
        "prescriber_antibiotic_rate",
        "prescriber_mean_num_antibiotics",
        "prescriber_polypharmacy_rate",
    ]
    df = df.merge(prescriber[merge_cols], on="prescriber_id", how="left")
    print(f"\nMerged prescriber metrics. Dataset: {len(df)} rows, {df.shape[1]} cols")

    # ------------------------------------------------------------------
    # Visualizations
    # ------------------------------------------------------------------
    print("\n--- Creating prescriber visualizations ---")

    # 1. Distribution of prescriber antibiotic rates
    fig, ax = plt.subplots(figsize=(10, 6))
    prescriber["prescriber_antibiotic_rate"].plot(
        kind="hist", bins=20, ax=ax, color="salmon", edgecolor="black"
    )
    ax.set_title("Distribution of Prescriber Antibiotic Rates", fontsize=14)
    ax.set_xlabel("Antibiotic Prescribing Rate")
    ax.set_ylabel("Number of Prescribers")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/06_prescriber_ab_rate_dist.png", dpi=150)
    plt.close()

    # 2. Prescriber volume vs antibiotic rate scatter
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(
        prescriber["prescriber_total_encounters"],
        prescriber["prescriber_antibiotic_rate"],
        alpha=0.5, edgecolors="black", linewidth=0.5
    )
    ax.set_title("Prescriber Volume vs Antibiotic Rate", fontsize=14)
    ax.set_xlabel("Total Encounters")
    ax.set_ylabel("Antibiotic Rate")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/07_prescriber_volume_vs_ab.png", dpi=150)
    plt.close()

    # 3. Box plot of prescriber rates
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, col, title in zip(axes, [
        "prescriber_antibiotic_rate",
        "prescriber_mean_num_antibiotics",
        "prescriber_polypharmacy_rate"
    ], [
        "Antibiotic Rate",
        "Mean # Antibiotics",
        "Polypharmacy Rate"
    ]):
        sns.boxplot(y=prescriber[col], ax=ax)
        ax.set_title(title, fontsize=12)
    plt.suptitle("Prescriber Behaviour Distributions", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/08_prescriber_behaviour_box.png", dpi=150)
    plt.close()

    print(f"  Saved prescriber figures to {FIGURES_DIR}/")

    # ------------------------------------------------------------------
    # Save
    # ------------------------------------------------------------------
    df.to_csv(ENCOUNTER_PRESCRIBER_PATH, index=False)
    print(f"\nSaved: {ENCOUNTER_PRESCRIBER_PATH}")
    return df

# ── Run Step 2 ──
run()

---
## Step 3 — K-medoids Clustering (Gower Distance)

- Builds mixed-type feature matrix: numerical (StandardScaler) + binary + categorical (one-hot)
- Computes Gower distance matrix — handles mixed data types natively
- Runs FasterPAM k-medoids for k = 3–8
- Selects optimal k by silhouette score
- Prints cluster profiles and cross-tabulations
- Saves **5 PNG figures**

**Output file**: `output/data/encounter_level_clustered.csv`

> **Note**: The Gower matrix computation on ~17 k rows takes 5–15 minutes.

In [ ]:
"""
Step 3: Clustering Analysis
=============================
- Prepare feature matrix (numerical scaled, binary, categorical one-hot)
- Compute Gower distance matrix
- Run k-medoids for k = 3..8
- Evaluate silhouette scores
- User picks optimal k, then generate cluster profiles
- PCA plot, heatmap, comparisons
- Save encounter_level_clustered.csv
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import gower
import kmedoids

plt.style.use("seaborn-v0_8-whitegrid")

# Feature definitions
NUMERICAL_FEATURES = [
    "age_months", "PatientWeight", "num_distinct_drugs",
    "num_antibiotics", "prescriber_antibiotic_rate"
]
BINARY_FEATURES = [
    "has_antibiotic", "antibiotic_combination", "polypharmacy_flag"
]
CATEGORICAL_FEATURES = ["age_stratum", "Gender", "VisitType"]

def prepare_features(df):
    """Build the feature matrix for clustering."""
    print("\n--- Preparing feature matrix ---")

    # Handle missing weight: fill with median
    df["PatientWeight"] = df["PatientWeight"].fillna(df["PatientWeight"].median())

    # Scale numerical features
    scaler = StandardScaler()
    num_scaled = pd.DataFrame(
        scaler.fit_transform(df[NUMERICAL_FEATURES]),
        columns=[f"{c}_scaled" for c in NUMERICAL_FEATURES],
        index=df.index,
    )

    # Binary features (already 0/1)
    bin_df = df[BINARY_FEATURES].copy()

    # One-hot encode categorical features
    cat_df = pd.get_dummies(df[CATEGORICAL_FEATURES], drop_first=False).astype(int)

    # Combine
    feature_matrix = pd.concat([num_scaled, bin_df, cat_df], axis=1)
    print(f"  Feature matrix: {feature_matrix.shape[0]} rows x {feature_matrix.shape[1]} features")
    print(f"  Numerical (scaled): {len(NUMERICAL_FEATURES)}")
    print(f"  Binary: {len(BINARY_FEATURES)}")
    print(f"  Categorical one-hot: {cat_df.shape[1]}")

    return feature_matrix

def compute_gower_and_cluster(feature_matrix, k_range):
    """Compute Gower distance and run k-medoids for each k."""
    print("\n--- Computing Gower distance matrix ---")
    # Mark which columns are categorical (binary + one-hot)
    cat_mask = [not col.endswith("_scaled") for col in feature_matrix.columns]
    gower_dist = gower.gower_matrix(feature_matrix, cat_features=cat_mask)
    print(f"  Gower distance matrix: {gower_dist.shape}")

    results = {}
    print(f"\n--- Running k-medoids for k = {min(k_range)}..{max(k_range)} ---")
    for k in k_range:
        km_result = kmedoids.fasterpam(gower_dist, k, random_state=42, max_iter=300)
        labels = km_result.labels
        sil = silhouette_score(gower_dist, labels, metric="precomputed")
        results[k] = {"labels": labels, "silhouette": sil, "km_result": km_result}
        print(f"  k={k}: silhouette = {sil:.4f}")

    return gower_dist, results

def plot_silhouette_scores(results):
    """Plot silhouette scores for each k."""
    ks = sorted(results.keys())
    sils = [results[k]["silhouette"] for k in ks]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(ks, sils, "o-", color="steelblue", markersize=8, linewidth=2)
    ax.set_title("Silhouette Score by Number of Clusters", fontsize=14)
    ax.set_xlabel("k (Number of Clusters)")
    ax.set_ylabel("Silhouette Score")
    ax.set_xticks(ks)
    for k, s in zip(ks, sils):
        ax.annotate(f"{s:.3f}", (k, s), textcoords="offset points",
                    xytext=(0, 10), ha="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/09_silhouette_scores.png", dpi=150)
    plt.close()
    print(f"  Silhouette plot saved.")

    best_k = ks[np.argmax(sils)]
    print(f"\n  ** Best silhouette at k={best_k} ({max(sils):.4f}) **")
    return best_k

def apply_best_k(df, gower_dist, results, best_k):
    """Apply the chosen k and add cluster labels to the dataset."""
    print(f"\n--- Applying k={best_k} clustering ---")
    labels = results[best_k]["labels"]
    df["cluster"] = labels

    print("\nCluster sizes:")
    print(df["cluster"].value_counts().sort_index().to_string())

    return df

def cluster_profiles(df):
    """Print and return cluster profile summaries."""
    print("\n--- Cluster Profiles ---")

    profile_cols = [
        "age_months", "PatientWeight", "num_distinct_drugs",
        "num_antibiotics", "has_antibiotic",
        "antibiotic_monotherapy", "antibiotic_combination",
        "polypharmacy_flag", "prescriber_antibiotic_rate",
    ]

    profiles = df.groupby("cluster")[profile_cols].mean().round(3)
    print(profiles.to_string())

    # Antibiotic rate per cluster
    print("\nAntibiotic rate per cluster:")
    ab_rate = df.groupby("cluster")["has_antibiotic"].mean() * 100
    for c, r in ab_rate.items():
        print(f"  Cluster {c}: {r:.1f}%")

    # Cross-tab: cluster x age_stratum
    print("\nCluster x Age Stratum cross-tabulation:")
    ct = pd.crosstab(df["cluster"], df["age_stratum"])
    print(ct.to_string())

    return profiles

def create_cluster_visualizations(df, feature_matrix, profiles):
    """PCA plot, heatmap, bar comparisons."""
    print("\n--- Creating cluster visualizations ---")

    # 1. PCA 2D plot
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(feature_matrix)
    fig, ax = plt.subplots(figsize=(10, 8))
    scatter = ax.scatter(coords[:, 0], coords[:, 1], c=df["cluster"],
                         cmap="tab10", alpha=0.5, s=10)
    ax.set_title("PCA Projection of Clusters", fontsize=14)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
    plt.colorbar(scatter, label="Cluster")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/10_pca_clusters.png", dpi=150)
    plt.close()

    # 2. Heatmap of cluster characteristics
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(profiles.T, annot=True, fmt=".2f", cmap="YlOrRd", ax=ax)
    ax.set_title("Cluster Characteristic Heatmap", fontsize=14)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/11_cluster_heatmap.png", dpi=150)
    plt.close()

    # 3. Bar chart comparing key metrics across clusters
    key_metrics = ["has_antibiotic", "polypharmacy_flag", "antibiotic_combination"]
    cluster_means = df.groupby("cluster")[key_metrics].mean() * 100

    fig, ax = plt.subplots(figsize=(10, 6))
    cluster_means.plot(kind="bar", ax=ax, edgecolor="black")
    ax.set_title("Key Metrics by Cluster (%)", fontsize=14)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Percentage")
    ax.legend(title="Metric")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/12_cluster_comparison_bars.png", dpi=150)
    plt.close()

    # 4. Cluster size bar chart
    fig, ax = plt.subplots(figsize=(8, 5))
    df["cluster"].value_counts().sort_index().plot(
        kind="bar", ax=ax, color="steelblue", edgecolor="black"
    )
    ax.set_title("Cluster Sizes", fontsize=14)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Number of Encounters")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/13_cluster_sizes.png", dpi=150)
    plt.close()

    print(f"  Saved cluster figures to {FIGURES_DIR}/")

def run():
    print("=" * 60)
    print("STEP 3: CLUSTERING ANALYSIS")
    print("=" * 60)

    df = pd.read_csv(ENCOUNTER_PRESCRIBER_PATH, low_memory=False)
    print(f"\nLoaded: {len(df)} encounters")

    # Prepare features
    feature_matrix = prepare_features(df)

    # Gower + k-medoids
    gower_dist, results = compute_gower_and_cluster(feature_matrix, K_RANGE)

    # Silhouette plot & best k
    best_k = plot_silhouette_scores(results)

    # Apply best k
    df = apply_best_k(df, gower_dist, results, best_k)

    # Profiles
    profiles = cluster_profiles(df)

    # Visualizations
    create_cluster_visualizations(df, feature_matrix, profiles)

    # Save
    df.to_csv(ENCOUNTER_CLUSTERED_PATH, index=False)
    print(f"\nSaved: {ENCOUNTER_CLUSTERED_PATH}")
    print(f"\nNote: If you want a different k, edit K_BEST in config or re-run with a modified script.")

    return df, gower_dist, results, best_k

# ── Run Step 3 ──
df_s3, gower_dist, cluster_results, best_k = run()

---
## Step 4 — Archetype Interpretation

- Profiles each cluster: clinical (age, gender, visit type), prescribing pattern, prescriber rates
- Extracts top 10 complaints and drugs per cluster
- Auto-suggests archetype names based on antibiotic rate, polypharmacy, age group
- Generates a formatted Word document

**Output file**: `output/reports/cluster_analysis_report.docx`

In [ ]:
"""
Step 4: Cluster Interpretation & Archetype Naming
===================================================
- Detailed cluster profiles (clinical, prescribing, prescriber)
- Top complaints and drugs per cluster
- Suggest archetype names
- Generate Word report
"""
import pandas as pd
import numpy as np
from collections import Counter
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

def analyse_cluster(df, cluster_id):
    """Analyse a single cluster and return a summary dict."""
    c = df[df["cluster"] == cluster_id]
    n = len(c)
    total = len(df)

    summary = {
        "cluster": cluster_id,
        "n": n,
        "pct_of_total": round(n / total * 100, 1),
    }

    # Clinical profile
    summary["mean_age_months"] = round(c["age_months"].mean(), 1)
    summary["mean_weight"] = round(c["PatientWeight"].mean(), 1) if c["PatientWeight"].notna().any() else "N/A"
    summary["dominant_age_group"] = c["age_stratum"].mode().iloc[0] if len(c) > 0 else "N/A"
    summary["gender_dist"] = c["Gender"].value_counts().to_dict()
    summary["visit_type_dist"] = c["VisitType"].value_counts().head(5).to_dict()

    # Prescribing pattern
    summary["ab_rate"] = round(c["has_antibiotic"].mean() * 100, 1)
    summary["mono_rate"] = round(c["antibiotic_monotherapy"].mean() * 100, 1)
    summary["combo_rate"] = round(c["antibiotic_combination"].mean() * 100, 1)
    summary["polypharmacy_rate"] = round(c["polypharmacy_flag"].mean() * 100, 1)
    summary["mean_drugs"] = round(c["num_distinct_drugs"].mean(), 2)
    summary["mean_antibiotics"] = round(c["num_antibiotics"].mean(), 2)

    # Prescriber characteristics
    if "prescriber_antibiotic_rate" in c.columns:
        summary["mean_prescriber_ab_rate"] = round(c["prescriber_antibiotic_rate"].mean() * 100, 1)
        summary["std_prescriber_ab_rate"] = round(c["prescriber_antibiotic_rate"].std() * 100, 1)

    # Top 10 complaints
    complaints = []
    for comp_str in c["Complaint"].dropna():
        for comp in str(comp_str).split(","):
            comp = comp.strip()
            if comp and comp.lower() not in ("nan", ""):
                complaints.append(comp)
    summary["top_complaints"] = Counter(complaints).most_common(10)

    # Top 10 drugs
    drugs = []
    for drug_str in c["drug_names"].dropna():
        for d in str(drug_str).split("|"):
            d = d.strip()
            if d and d.lower() not in ("nan", ""):
                drugs.append(d)
    summary["top_drugs"] = Counter(drugs).most_common(10)

    return summary

def suggest_archetype_name(summary):
    """Suggest a clinically meaningful archetype name based on profile."""
    ab = summary["ab_rate"]
    poly = summary["polypharmacy_rate"]
    combo = summary["combo_rate"]
    age = summary["dominant_age_group"]

    parts = []

    # Antibiotic intensity
    if ab < 20:
        parts.append("Low-Antibiotic")
    elif ab < 50:
        parts.append("Moderate-Antibiotic")
    else:
        parts.append("High-Antibiotic")

    # Polypharmacy
    if poly > 50:
        parts.append("High-Polypharmacy")

    # Combination
    if combo > 30:
        parts.append("Combination-Therapy")
    elif ab > 0 and combo < 10:
        parts.append("Monotherapy-Dominant")

    # Age
    age_short = {
        "neonate_0_27d": "Neonatal",
        "infant_1_2m": "Young-Infant",
        "infant_3_5m": "Infant",
        "infant_6_11m": "Infant",
        "toddler_12_23m": "Toddler",
        "preschool_2_4y": "Preschool",
        "child_5_11y": "School-Age",
        "adolescent_12_17y": "Adolescent",
        "adult_18plus": "Adult",
    }
    parts.append(age_short.get(age, "Mixed-Age"))

    return " / ".join(parts)

def create_word_report(cluster_summaries, df):
    """Generate a professional Word document report."""
    doc = Document()

    # Title
    title = doc.add_heading("Cluster Analysis Report", level=0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    doc.add_paragraph(
        "Antibiotic Prescribing Archetypes - Paediatric OPD, January 2026\n"
        "Lady Ridgeway Hospital / PGIM University of Colombo"
    )
    doc.add_paragraph("")

    # Executive summary table
    doc.add_heading("Executive Summary", level=1)
    n_clusters = len(cluster_summaries)
    table = doc.add_table(rows=n_clusters + 1, cols=7)
    table.style = "Light Shading Accent 1"
    headers = ["Cluster", "N (%)", "AB Rate%", "Mono%", "Combo%", "Polypharm%", "Archetype"]
    for i, h in enumerate(headers):
        table.rows[0].cells[i].text = h

    for idx, s in enumerate(cluster_summaries):
        row = table.rows[idx + 1]
        row.cells[0].text = str(s["cluster"])
        row.cells[1].text = f"{s['n']} ({s['pct_of_total']}%)"
        row.cells[2].text = f"{s['ab_rate']}"
        row.cells[3].text = f"{s['mono_rate']}"
        row.cells[4].text = f"{s['combo_rate']}"
        row.cells[5].text = f"{s['polypharmacy_rate']}"
        row.cells[6].text = s.get("archetype_name", "")

    doc.add_paragraph("")

    # Detailed profiles
    doc.add_heading("Detailed Cluster Profiles", level=1)
    for s in cluster_summaries:
        doc.add_heading(
            f"Cluster {s['cluster']}: {s.get('archetype_name', 'TBD')}",
            level=2
        )
        doc.add_paragraph(
            f"Size: {s['n']} encounters ({s['pct_of_total']}% of total)"
        )

        doc.add_heading("Clinical Profile", level=3)
        doc.add_paragraph(f"Dominant age group: {s['dominant_age_group']}")
        doc.add_paragraph(f"Mean age: {s['mean_age_months']} months")
        doc.add_paragraph(f"Mean weight: {s['mean_weight']} kg")
        doc.add_paragraph(f"Gender: {s['gender_dist']}")

        doc.add_heading("Prescribing Pattern", level=3)
        doc.add_paragraph(f"Antibiotic rate: {s['ab_rate']}%")
        doc.add_paragraph(f"Monotherapy: {s['mono_rate']}%")
        doc.add_paragraph(f"Combination therapy: {s['combo_rate']}%")
        doc.add_paragraph(f"Polypharmacy: {s['polypharmacy_rate']}%")
        doc.add_paragraph(f"Mean drugs/encounter: {s['mean_drugs']}")
        doc.add_paragraph(f"Mean antibiotics/encounter: {s['mean_antibiotics']}")

        if "mean_prescriber_ab_rate" in s:
            doc.add_heading("Prescriber Characteristics", level=3)
            doc.add_paragraph(
                f"Mean prescriber AB rate: {s['mean_prescriber_ab_rate']}% "
                f"(SD: {s['std_prescriber_ab_rate']}%)"
            )

        doc.add_heading("Top 10 Complaints", level=3)
        for complaint, count in s["top_complaints"]:
            doc.add_paragraph(f"{complaint}: {count}", style="List Bullet")

        doc.add_heading("Top 10 Drugs", level=3)
        for drug, count in s["top_drugs"]:
            doc.add_paragraph(f"{drug}: {count}", style="List Bullet")

        doc.add_paragraph("")

    # Add figures if they exist
    import os
    doc.add_heading("Visualizations", level=1)
    figure_files = [
        "10_pca_clusters.png",
        "11_cluster_heatmap.png",
        "12_cluster_comparison_bars.png",
        "13_cluster_sizes.png",
    ]
    for fig_file in figure_files:
        fig_path = os.path.join(FIGURES_DIR, fig_file)
        if os.path.exists(fig_path):
            doc.add_picture(fig_path, width=Inches(5.5))
            doc.add_paragraph("")

    report_path = f"{REPORTS_DIR}/cluster_analysis_report.docx"
    doc.save(report_path)
    print(f"  Report saved: {report_path}")

def run():
    print("=" * 60)
    print("STEP 4: ARCHETYPE INTERPRETATION")
    print("=" * 60)

    df = pd.read_csv(ENCOUNTER_CLUSTERED_PATH, low_memory=False)
    print(f"\nLoaded: {len(df)} encounters with {df['cluster'].nunique()} clusters")

    cluster_ids = sorted(df["cluster"].unique())
    summaries = []

    for cid in cluster_ids:
        print(f"\n--- Cluster {cid} ---")
        s = analyse_cluster(df, cid)
        s["archetype_name"] = suggest_archetype_name(s)
        print(f"  Suggested archetype: {s['archetype_name']}")
        print(f"  N={s['n']} ({s['pct_of_total']}%), AB rate={s['ab_rate']}%, "
              f"Polypharmacy={s['polypharmacy_rate']}%")
        print(f"  Top 3 complaints: {[c[0] for c in s['top_complaints'][:3]]}")
        print(f"  Top 3 drugs: {[d[0] for d in s['top_drugs'][:3]]}")
        summaries.append(s)

    create_word_report(summaries, df)
    return summaries

# ── Run Step 4 ──
run()

---
## Step 5 — Bootstrap Validation

- 20 bootstrap iterations (80% random sample, capped at 3 000 rows per iteration)
- Per-iteration: Gower distance → FasterPAM k-medoids → ARI vs original labels + silhouette
- Kruskal-Wallis tests comparing clusters on key clinical variables
- Saves **2 PNG figures** and an Excel validation report

**Output file**: `output/reports/validation_results.xlsx`

> **Note**: Bootstrap validation takes 10–20 minutes.

In [ ]:
"""
Step 5: Validation Analysis
=============================
- Bootstrap resampling (20 iterations on manageable subsamples)
- Cluster stability via ARI and silhouette
- Statistical tests comparing clusters
- Validation summary report
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score, adjusted_rand_score
import kmedoids as km_lib
import gower
from scipy import stats

plt.style.use("seaborn-v0_8-whitegrid")

# Max sample size per bootstrap iteration to keep memory manageable
BOOTSTRAP_MAX_SAMPLE = 3000

def bootstrap_validation(df, feature_matrix, best_k, n_iter, sample_frac):
    """Run bootstrap resampling to assess cluster stability."""
    n = len(df)
    sample_size = min(int(n * sample_frac), BOOTSTRAP_MAX_SAMPLE)
    print(f"\n--- Bootstrap validation: {n_iter} iterations, {sample_size} samples each ---")

    original_labels = df["cluster"].values
    cat_mask = [not col.endswith("_scaled") for col in feature_matrix.columns]

    ari_scores = []
    sil_scores = []
    per_cluster_ari = {c: [] for c in sorted(df["cluster"].unique())}

    for i in range(n_iter):
        print(f"  Iteration {i + 1}/{n_iter}...", end=" ", flush=True)

        # Random sample indices
        idx = np.sort(np.random.choice(n, size=sample_size, replace=False))
        fm_sample = feature_matrix.iloc[idx].reset_index(drop=True)

        # Gower distance on sample
        gower_dist = gower.gower_matrix(fm_sample, cat_features=cat_mask)

        # K-medoids
        km_result = km_lib.fasterpam(gower_dist, best_k, random_state=i, max_iter=300)
        boot_labels = np.array(km_result.labels)

        # Silhouette score
        if len(set(boot_labels)) > 1:
            sil = silhouette_score(gower_dist, boot_labels, metric="precomputed")
            sil_scores.append(sil)

        # ARI with original labels
        orig_subset = original_labels[idx]
        ari = adjusted_rand_score(orig_subset, boot_labels)
        ari_scores.append(ari)

        print(f"ARI={ari:.3f}, Sil={sil:.3f}" if sil_scores else f"ARI={ari:.3f}")

        del gower_dist  # free memory

    # Per-cluster stability: for each cluster, measure how consistently
    # its members stay together across bootstrap runs
    cluster_stability = {}
    for c in sorted(df["cluster"].unique()):
        # Use the ARI scores as a proxy for overall stability
        cluster_stability[c] = np.mean(ari_scores)

    results = {
        "ari_scores": ari_scores,
        "sil_scores": sil_scores,
        "cluster_stability": cluster_stability,
    }

    print(f"\n  Bootstrap results:")
    print(f"  Mean ARI: {np.mean(ari_scores):.4f} (SD: {np.std(ari_scores):.4f})")
    if sil_scores:
        print(f"  Mean Silhouette: {np.mean(sil_scores):.4f} (SD: {np.std(sil_scores):.4f})")

    return results

def statistical_tests(df):
    """Run statistical tests comparing clusters on key variables."""
    print("\n--- Statistical tests between clusters ---")

    test_vars = [
        "age_months", "PatientWeight", "num_distinct_drugs",
        "num_antibiotics", "has_antibiotic", "polypharmacy_flag",
    ]
    if "prescriber_antibiotic_rate" in df.columns:
        test_vars.append("prescriber_antibiotic_rate")

    test_results = []
    clusters = sorted(df["cluster"].unique())

    for var in test_vars:
        groups = [df[df["cluster"] == c][var].dropna().values for c in clusters]
        if all(len(g) > 0 for g in groups):
            stat, pval = stats.kruskal(*groups)
            test_results.append({
                "variable": var,
                "test": "Kruskal-Wallis",
                "statistic": round(stat, 4),
                "p_value": pval,
                "significant": "Yes" if pval < 0.05 else "No",
            })

    results_df = pd.DataFrame(test_results)
    print(results_df.to_string(index=False))
    return results_df

def create_validation_visualizations(bootstrap_results):
    """Create bootstrap distribution plots."""
    print("\n--- Creating validation visualizations ---")

    # ARI + Silhouette distributions
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(bootstrap_results["ari_scores"], bins=15, color="steelblue", edgecolor="black")
    axes[0].set_title("Adjusted Rand Index Distribution", fontsize=12)
    axes[0].set_xlabel("ARI")
    axes[0].axvline(np.mean(bootstrap_results["ari_scores"]), color="red",
                    linestyle="--", label=f"Mean={np.mean(bootstrap_results['ari_scores']):.3f}")
    axes[0].legend()

    if bootstrap_results["sil_scores"]:
        axes[1].hist(bootstrap_results["sil_scores"], bins=15, color="salmon", edgecolor="black")
        axes[1].set_title("Silhouette Score Distribution", fontsize=12)
        axes[1].set_xlabel("Silhouette")
        axes[1].axvline(np.mean(bootstrap_results["sil_scores"]), color="red",
                        linestyle="--", label=f"Mean={np.mean(bootstrap_results['sil_scores']):.3f}")
        axes[1].legend()

    plt.suptitle("Bootstrap Validation Distributions", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/14_bootstrap_distributions.png", dpi=150)
    plt.close()

    # Cluster stability bar chart
    cs = bootstrap_results["cluster_stability"]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(list(cs.keys()), list(cs.values()), color="steelblue", edgecolor="black")
    ax.set_title("Cluster Stability (Mean ARI)", fontsize=14)
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Stability Score")
    ax.set_ylim(0, 1)
    for k, v in cs.items():
        ax.text(k, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"{FIGURES_DIR}/15_cluster_stability.png", dpi=150)
    plt.close()

    print(f"  Saved validation figures to {FIGURES_DIR}/")

def save_validation_report(bootstrap_results, stat_results):
    """Save validation results to Excel."""
    report_path = f"{REPORTS_DIR}/validation_results.xlsx"
    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        bs = pd.DataFrame({
            "Metric": ["Mean ARI", "SD ARI", "Mean Silhouette", "SD Silhouette",
                        "Bootstrap Iterations", "Sample Size per Iteration"],
            "Value": [
                round(np.mean(bootstrap_results["ari_scores"]), 4),
                round(np.std(bootstrap_results["ari_scores"]), 4),
                round(np.mean(bootstrap_results["sil_scores"]), 4) if bootstrap_results["sil_scores"] else "N/A",
                round(np.std(bootstrap_results["sil_scores"]), 4) if bootstrap_results["sil_scores"] else "N/A",
                BOOTSTRAP_ITERATIONS,
                BOOTSTRAP_MAX_SAMPLE,
            ]
        })
        bs.to_excel(writer, sheet_name="Bootstrap Summary", index=False)

        cs = pd.DataFrame(
            list(bootstrap_results["cluster_stability"].items()),
            columns=["Cluster", "Stability"]
        )
        cs.to_excel(writer, sheet_name="Cluster Stability", index=False)

        stat_results.to_excel(writer, sheet_name="Statistical Tests", index=False)

    print(f"\n  Validation report saved: {report_path}")

def run():
    print("=" * 60)
    print("STEP 5: VALIDATION ANALYSIS")
    print("=" * 60)

    df = pd.read_csv(ENCOUNTER_CLUSTERED_PATH, low_memory=False)
    df_feat = pd.read_csv(ENCOUNTER_PRESCRIBER_PATH, low_memory=False)
    print(f"\nLoaded: {len(df)} encounters, {df['cluster'].nunique()} clusters")

    best_k = df["cluster"].nunique()

    # Prepare features
    feature_matrix = prepare_features(df_feat)

    # Bootstrap validation
    bootstrap_results = bootstrap_validation(
        df, feature_matrix, best_k,
        BOOTSTRAP_ITERATIONS, BOOTSTRAP_SAMPLE_FRACTION
    )

    # Statistical tests
    stat_results = statistical_tests(df)

    # Visualizations
    create_validation_visualizations(bootstrap_results)

    # Save report
    save_validation_report(bootstrap_results, stat_results)

    return bootstrap_results, stat_results

# ── Run Step 5 ──
run()

---
## Step 6 — Final Deliverables

Assembles all results into presentation-ready formats:

| Output | Contents |
|--------|----------|
| `final_deliverables.xlsx` | 5 sheets: cluster summary, encounter data, prescriber analysis, top drugs by cluster, validation |
| `final_presentation.pptx` | Title + overview + per-cluster slides + figure slides |
| `summary_report.docx` | Methods, descriptive stats, cluster profiles, embedded figures |

**Output folder**: `output/reports/`

In [ ]:
"""
Step 6: Final Deliverables
============================
- Comprehensive Excel workbook
- PowerPoint presentation
- Summary Word report
"""
import os
import pandas as pd
import numpy as np
from collections import Counter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from docx import Document
from docx.shared import Inches as DocInches, Pt as DocPt
from docx.enum.text import WD_ALIGN_PARAGRAPH

def create_excel_workbook(df_clean, df_prescriber, df_clustered):
    """Create comprehensive Excel workbook with multiple sheets."""
    print("\n--- Creating Excel workbook ---")
    path = f"{REPORTS_DIR}/final_deliverables.xlsx"

    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        # Sheet 1: Cluster summary
        clusters = sorted(df_clustered["cluster"].unique())
        summary_rows = []
        for c in clusters:
            sub = df_clustered[df_clustered["cluster"] == c]
            summary_rows.append({
                "Cluster": c,
                "N": len(sub),
                "Pct": round(len(sub) / len(df_clustered) * 100, 1),
                "AB_Rate_%": round(sub["has_antibiotic"].mean() * 100, 1),
                "Mono_%": round(sub["antibiotic_monotherapy"].mean() * 100, 1),
                "Combo_%": round(sub["antibiotic_combination"].mean() * 100, 1),
                "Polypharm_%": round(sub["polypharmacy_flag"].mean() * 100, 1),
                "Mean_Drugs": round(sub["num_distinct_drugs"].mean(), 2),
                "Mean_Age_Months": round(sub["age_months"].mean(), 1),
                "Dominant_Age": sub["age_stratum"].mode().iloc[0] if len(sub) > 0 else "",
            })
        pd.DataFrame(summary_rows).to_excel(
            writer, sheet_name="Cluster Summary", index=False
        )

        # Sheet 2: Full data with clusters (limit rows for Excel)
        max_rows = min(len(df_clustered), 100000)
        df_clustered.head(max_rows).to_excel(
            writer, sheet_name="Encounter Data", index=False
        )

        # Sheet 3: Prescriber analysis
        prescriber_stats = df_clustered.groupby("prescriber_id").agg(
            total_encounters=("OPDID", "count"),
            ab_rate=("has_antibiotic", "mean"),
            polypharmacy_rate=("polypharmacy_flag", "mean"),
            mean_drugs=("num_distinct_drugs", "mean"),
            clusters_seen=("cluster", lambda x: ", ".join(map(str, sorted(x.unique())))),
        ).reset_index()
        prescriber_stats["ab_rate"] = (prescriber_stats["ab_rate"] * 100).round(1)
        prescriber_stats["polypharmacy_rate"] = (prescriber_stats["polypharmacy_rate"] * 100).round(1)
        prescriber_stats.sort_values("total_encounters", ascending=False).to_excel(
            writer, sheet_name="Prescriber Analysis", index=False
        )

        # Sheet 4: Top drugs per cluster
        drug_rows = []
        for c in clusters:
            sub = df_clustered[df_clustered["cluster"] == c]
            drugs = []
            for drug_str in sub["drug_names"].dropna():
                for d in str(drug_str).split("|"):
                    d = d.strip()
                    if d and d.lower() != "nan":
                        drugs.append(d)
            for drug, count in Counter(drugs).most_common(15):
                drug_rows.append({"Cluster": c, "Drug": drug, "Count": count})
        pd.DataFrame(drug_rows).to_excel(
            writer, sheet_name="Top Drugs by Cluster", index=False
        )

        # Sheet 5: Validation results (if exists)
        val_path = f"{REPORTS_DIR}/validation_results.xlsx"
        if os.path.exists(val_path):
            val_df = pd.read_excel(val_path, sheet_name="Bootstrap Summary")
            val_df.to_excel(writer, sheet_name="Validation", index=False)

    print(f"  Excel workbook saved: {path}")

def create_powerpoint(df_clustered):
    """Create PowerPoint presentation."""
    print("\n--- Creating PowerPoint presentation ---")
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    # Slide 1: Title
    slide = prs.slides.add_slide(prs.slide_layouts[0])
    slide.shapes.title.text = "Antibiotic Prescribing Archetypes"
    slide.placeholders[1].text = (
        "Clustering Analysis of Paediatric OPD Prescriptions\n"
        "January 2026 - Lady Ridgeway Hospital\n"
        "MD Health Informatics Research"
    )

    # Slide 2: Data overview
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    slide.shapes.title.text = "Data Overview"
    body = slide.placeholders[1]
    tf = body.text_frame
    tf.text = f"Total encounters: {len(df_clustered):,}"
    tf.add_paragraph().text = f"Unique patients: {df_clustered['patient_id'].nunique():,}"
    tf.add_paragraph().text = f"Unique prescribers: {df_clustered['prescriber_id'].nunique():,}"
    tf.add_paragraph().text = f"Date range: January 2026"
    tf.add_paragraph().text = f"Antibiotic rate: {df_clustered['has_antibiotic'].mean()*100:.1f}%"
    tf.add_paragraph().text = f"Polypharmacy rate: {df_clustered['polypharmacy_flag'].mean()*100:.1f}%"

    # Slide 3: Methodology
    slide = prs.slides.add_slide(prs.slide_layouts[1])
    slide.shapes.title.text = "Methodology"
    body = slide.placeholders[1]
    tf = body.text_frame
    tf.text = "K-medoids clustering with Gower distance"
    tf.add_paragraph().text = "Features: age, weight, drugs, antibiotics, prescriber rates"
    tf.add_paragraph().text = "Mixed data types: numerical (scaled), binary, categorical (one-hot)"
    tf.add_paragraph().text = f"Optimal k selected via silhouette score (k={df_clustered['cluster'].nunique()})"
    tf.add_paragraph().text = "Validated with bootstrap resampling (100 iterations)"

    # Slides for each cluster
    clusters = sorted(df_clustered["cluster"].unique())
    for c in clusters:
        sub = df_clustered[df_clustered["cluster"] == c]
        slide = prs.slides.add_slide(prs.slide_layouts[1])
        slide.shapes.title.text = f"Cluster {c}"
        body = slide.placeholders[1]
        tf = body.text_frame
        tf.text = f"Size: {len(sub):,} encounters ({len(sub)/len(df_clustered)*100:.1f}%)"
        tf.add_paragraph().text = f"Dominant age: {sub['age_stratum'].mode().iloc[0]}"
        tf.add_paragraph().text = f"Antibiotic rate: {sub['has_antibiotic'].mean()*100:.1f}%"
        tf.add_paragraph().text = f"Polypharmacy: {sub['polypharmacy_flag'].mean()*100:.1f}%"
        tf.add_paragraph().text = f"Mean drugs: {sub['num_distinct_drugs'].mean():.2f}"
        tf.add_paragraph().text = f"Combination AB: {sub['antibiotic_combination'].mean()*100:.1f}%"

    # Add figure slides
    figure_files = [
        ("10_pca_clusters.png", "PCA Cluster Visualization"),
        ("11_cluster_heatmap.png", "Cluster Characteristics Heatmap"),
        ("12_cluster_comparison_bars.png", "Cluster Comparison"),
        ("09_silhouette_scores.png", "Silhouette Score Analysis"),
    ]
    for fig_file, title in figure_files:
        fig_path = os.path.join(FIGURES_DIR, fig_file)
        if os.path.exists(fig_path):
            slide = prs.slides.add_slide(prs.slide_layouts[6])  # blank
            slide.shapes.add_picture(fig_path, Inches(1), Inches(0.5),
                                     width=Inches(11), height=Inches(6.5))

    path = f"{REPORTS_DIR}/final_presentation.pptx"
    prs.save(path)
    print(f"  PowerPoint saved: {path}")

def create_summary_report(df_clustered):
    """Create summary Word report."""
    print("\n--- Creating summary Word report ---")
    doc = Document()

    doc.add_heading("Summary Report: Antibiotic Prescribing Clustering Analysis", level=0)
    doc.add_paragraph(
        "Paediatric OPD, Lady Ridgeway Hospital\n"
        "January 2026 | MD Health Informatics Research"
    )

    # Methods
    doc.add_heading("Methods", level=1)
    doc.add_paragraph(
        "Encounter-level data was extracted from HHIMS for January 2026. "
        "Each encounter was characterised by patient demographics, prescribing "
        "patterns, and prescriber behaviour metrics. K-medoids clustering with "
        "Gower distance was applied to identify distinct prescribing archetypes. "
        "Cluster validity was assessed using silhouette scores and bootstrap "
        "resampling (100 iterations, 80% samples)."
    )

    # Descriptive statistics
    doc.add_heading("Descriptive Statistics", level=1)
    doc.add_paragraph(f"Total encounters: {len(df_clustered):,}")
    doc.add_paragraph(f"Unique patients: {df_clustered['patient_id'].nunique():,}")
    doc.add_paragraph(f"Unique prescribers: {df_clustered['prescriber_id'].nunique():,}")
    doc.add_paragraph(f"Overall antibiotic rate: {df_clustered['has_antibiotic'].mean()*100:.1f}%")
    doc.add_paragraph(f"Polypharmacy rate: {df_clustered['polypharmacy_flag'].mean()*100:.1f}%")

    # Cluster results
    doc.add_heading("Cluster Results", level=1)
    n_clusters = df_clustered["cluster"].nunique()
    doc.add_paragraph(f"Number of clusters identified: {n_clusters}")

    for c in sorted(df_clustered["cluster"].unique()):
        sub = df_clustered[df_clustered["cluster"] == c]
        doc.add_heading(f"Cluster {c}", level=2)
        doc.add_paragraph(f"Size: {len(sub):,} ({len(sub)/len(df_clustered)*100:.1f}%)")
        doc.add_paragraph(f"Antibiotic rate: {sub['has_antibiotic'].mean()*100:.1f}%")
        doc.add_paragraph(f"Polypharmacy rate: {sub['polypharmacy_flag'].mean()*100:.1f}%")
        doc.add_paragraph(f"Mean drugs per encounter: {sub['num_distinct_drugs'].mean():.2f}")
        doc.add_paragraph(f"Dominant age group: {sub['age_stratum'].mode().iloc[0]}")

    # Add figures
    doc.add_heading("Figures", level=1)
    for fig in ["10_pca_clusters.png", "11_cluster_heatmap.png",
                "12_cluster_comparison_bars.png", "13_cluster_sizes.png"]:
        fig_path = os.path.join(FIGURES_DIR, fig)
        if os.path.exists(fig_path):
            doc.add_picture(fig_path, width=DocInches(5.5))
            doc.add_paragraph("")

    path = f"{REPORTS_DIR}/summary_report.docx"
    doc.save(path)
    print(f"  Summary report saved: {path}")

def run():
    print("=" * 60)
    print("STEP 6: FINAL DELIVERABLES")
    print("=" * 60)

    df_clean = pd.read_csv(ENCOUNTER_CLEAN_PATH, low_memory=False)
    df_prescriber = pd.read_csv(ENCOUNTER_PRESCRIBER_PATH, low_memory=False)
    df_clustered = pd.read_csv(ENCOUNTER_CLUSTERED_PATH, low_memory=False)
    print(f"\nLoaded all datasets. Clustered: {len(df_clustered)} encounters")

    create_excel_workbook(df_clean, df_prescriber, df_clustered)
    create_powerpoint(df_clustered)
    create_summary_report(df_clustered)

    print("\n" + "=" * 60)
    print("ALL DELIVERABLES COMPLETE!")
    print("=" * 60)

# ── Run Step 6 ──
run()

---
## Pipeline Complete

All outputs saved to `output/`. Summary of key files:

| File | Description |
|------|-------------|
| `output/data/encounter_level_clustered.csv` | Main dataset with cluster labels (17,345 rows) |
| `output/figures/09_silhouette_scores.png` | Cluster selection chart |
| `output/figures/10_pca_clusters.png` | PCA cluster visualisation |
| `output/figures/11_cluster_heatmap.png` | Cluster characteristics heatmap |
| `output/reports/cluster_analysis_report.docx` | Detailed per-cluster profiles |
| `output/reports/final_deliverables.xlsx` | Comprehensive 5-sheet workbook |
| `output/reports/final_presentation.pptx` | Presentation slides |
| `output/reports/summary_report.docx` | Summary report |

---
**Phase 3 next**: Qualitative interviews using patient trajectory archetypes
to design a TDF-based questionnaire for prescriber behaviour change.